# Part 1: Data Cleaning & Exploratory Data Analysis (EDA)
## Sales & Demand Forecasting for Businesses
**Internship Task 1 (Future Interns)**  

### Objective:
In this notebook, we load the raw transactional retail sales dataset, clean missing or irregular values, parse datetime records, and explore patterns in sales volume, weekly cycles, and monthly seasonality.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["font.size"] = 10

### 1. Load Raw Superstore Dataset

In [ ]:
from src.data_loader import load_raw_superstore

raw_path = os.path.join("..", "data", "raw", "superstore.csv")
df_raw = load_raw_superstore(raw_path)
print(f"Raw dataset shape: {df_raw.shape}")
df_raw.head()

### 2. Data Cleaning & Type Formatting
- Normalize column headers to lowercase with underscores
- Parse dates into ISO datetime format
- Verify absence of non-positive transaction amounts
- Sort records in chronological sequence

In [ ]:
from src.data_loader import clean_sales_data

df_clean = clean_sales_data(df_raw)
print(f"Cleaned transaction records: {len(df_clean):,}")
df_clean.info()

### 3. Continuous Daily Resampling
Retail time series require contiguous calendar dates. Missing trading dates are filled with `0` sales to preserve temporal integrity.

In [ ]:
from src.data_loader import aggregate_daily_sales

daily_df = aggregate_daily_sales(df_clean)
print(f"Daily timeline covers: {daily_df['order_date'].min().strftime('%Y-%m-%d')} to {daily_df['order_date'].max().strftime('%Y-%m-%d')}")
daily_df.head(10)

### 4. Historical Trend Visualizations

In [ ]:
daily_df['rolling_30'] = daily_df['total_sales'].rolling(30, min_periods=7).mean()

plt.figure(figsize=(14, 5))
plt.plot(daily_df['order_date'], daily_df['total_sales'], color='#90CAF9', alpha=0.5, label='Actual Daily Sales')
plt.plot(daily_df['order_date'], daily_df['rolling_30'], color='#1565C0', linewidth=2.5, label='30-Day Moving Average')
plt.title('Daily Store Sales Volume Over Time', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Sales ($)')
plt.legend()
plt.tight_layout()
plt.show()

### 5. Seasonality: Monthly Breakdown
Observe high-demand quarters (specifically Q4 retail holidays).

In [ ]:
df_clean['month_name'] = df_clean['order_date'].dt.strftime('%b')
df_clean['month_num'] = df_clean['order_date'].dt.month
monthly = df_clean.groupby(['month_num', 'month_name'])['sales'].sum().reset_index().sort_values('month_num')

plt.figure(figsize=(10, 5))
sns.barplot(data=monthly, x='month_name', y='sales', palette='Blues_d')
plt.title('Aggregated Sales by Calendar Month', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.tight_layout()
plt.show()

### 6. Day of Week Patterns

In [ ]:
df_clean['day_name'] = df_clean['order_date'].dt.day_name()
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow = df_clean.groupby('day_name')['sales'].mean().reindex(order).reset_index()

plt.figure(figsize=(9, 4))
sns.barplot(data=dow, x='day_name', y='sales', palette='crest')
plt.title('Average Order Value by Day of Week', fontsize=14, fontweight='bold')
plt.xlabel('Day of Week')
plt.ylabel('Mean Order Value ($)')
plt.tight_layout()
plt.show()